# Phase 3 — Independent statistical audit for epoxy-composite thermal conductivity

**Manuscript:** MEAS-D-26-20648 · **Purpose:** reproducibly verify the completed Phase 2 Kaggle outputs before revising the paper.

**Input:** the *final* `TC_BLOCK2_RESULTS.zip` produced by `TC_PHASE2_FINAL_PLUS_TABICLV2_HIGH_EXPLORATORY.ipynb`. Upload it under **Kaggle → Add Input** as a dataset. No original CSVs, source notebook or model weights are required for this audit because the archive includes its original data, predictions and reports.

**No models are trained here. No predictions are fabricated, replaced or edited.** We recompute scores from recorded held-out predictions; audit folds, data provenance, augmentation, statistical tests, uncertainty, and physical-evidence gates; and export **`TC_PHASE3_STATISTICAL_AUDIT.zip`**.

**Evidence rules:** Source labels have NOT been independently paper-to-row verified. Therefore publication-labeled CV is a provisional software sensitivity study, not independently validated source transfer. High-temperature composition-signature CV is exploratory rather than independent publication validation. Lack of laboratory measurement records cannot be solved statistically.

**Reviewers:** R1-1–5, 8–10, 12, 15–16; R2-1–3, 7, 9–11, 13–17; R3-2–5 are mapped to the relevant checks. All 40 comments remain in the supplied reviewer register, including non-statistical items that are out of scope for Phase 3.

## 1. Configuration, imports, and input discovery

Keep `INPUT_ZIP = None` for automatic discovery when exactly one complete Phase 2 archive is attached. If Kaggle has multiple Phase 2 ZIPs, set `INPUT_ZIP` to the exact path shown in Kaggle's Input sidebar. Internet and GPU are not required.

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import product, combinations
from datetime import datetime, timezone
import hashlib, io, itertools, json, math, os, platform, re, sys, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


INPUT_ZIP = os.environ.get("TC_PHASE2_ARCHIVE", "/kaggle/input/notebooks/shanmukhadattaboda/epoxy-thermal-conducitviy-ml-training/TC_BLOCK2_RESULTS.zip")
OUTPUT_BASE = Path(os.environ.get('TC_PHASE3_OUTPUT_BASE', '/kaggle/working' if Path('/kaggle/working').exists() else os.getcwd()))
OUT = OUTPUT_BASE / 'TC_PHASE3_STATISTICAL_AUDIT'
if OUT.exists() and any(OUT.iterdir()) and os.environ.get('TC_OVERWRITE_RESULTS', '0') != '1':
    raise FileExistsError(f'{OUT} contains files. Archive it first; set TC_OVERWRITE_RESULTS=1 only for intentional replacement.')
if OUT.exists() and os.environ.get('TC_OVERWRITE_RESULTS', '0') == '1':
    import shutil
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
SEED = 20261108
RNG = np.random.default_rng(SEED)
EPS = 1e-10
EXPECTED_ROWS = {'cryogenic':54, 'room':160, 'high':42}
EXPECTED_FILES = [
 '00_config/RUN_MANIFEST.json', '11_reports/ARTIFACT_INDEX.csv',
 '01_audit/all_256_original_rows_unchanged.csv',
 '03_splits/publication_outer_split_manifest.csv',
 '05_nested_cv/outer_predictions.csv',
 '05_nested_cv/tabicl_oof_predictions.csv',
 '05_nested_cv/high_composition_exploratory_predictions.csv',
 '11_reports/REVIEWER_40_ISSUE_STATUS.csv'
]

class Phase2Archive:
    def __init__(self, path):
        self.path = Path(path)
        self.is_zip = self.path.is_file()
        self.zip = zipfile.ZipFile(self.path, 'r') if self.is_zip else None
        names = self.zip.namelist() if self.is_zip else [x.relative_to(self.path).as_posix() for x in self.path.rglob('*') if x.is_file()]
        candidates = sorted(set(n[:-len('00_config/RUN_MANIFEST.json')] for n in names if n.endswith('00_config/RUN_MANIFEST.json')))
        if len(candidates) != 1: raise RuntimeError(f'Expected one Phase2 package root; found {candidates}')
        self.prefix = candidates[0]
    def exists(self, rel):
        key = self.prefix+rel
        return (key in self.zip.namelist()) if self.is_zip else (self.path/key).is_file()
    def read(self, rel):
        if not self.exists(rel): raise FileNotFoundError(f'Missing Phase2 artifact: {rel}')
        return self.zip.read(self.prefix+rel) if self.is_zip else (self.path/self.prefix/rel).read_bytes()
    def csv(self, rel, allow_empty=False):
        b=self.read(rel)
        try: return pd.read_csv(io.BytesIO(b))
        except pd.errors.EmptyDataError:
            if allow_empty: return pd.DataFrame()
            raise RuntimeError(f'Unexpected empty CSV: {rel}')
    def json(self, rel): return json.loads(self.read(rel).decode('utf-8'))
    def list(self):
        if self.is_zip: return [s[len(self.prefix):] for s in self.zip.namelist() if s.startswith(self.prefix) and not s.endswith('/')]
        return [x.relative_to(self.path/self.prefix).as_posix() for x in (self.path/self.prefix).rglob('*') if x.is_file()]

def is_phase2_candidate(path):
    try:
        z=Phase2Archive(path)
        return all(z.exists(x) for x in EXPECTED_FILES)
    except Exception: return False

def locate_phase2():
    if INPUT_ZIP:
        p=Path(INPUT_ZIP)
        if not p.exists(): raise FileNotFoundError(f'Configured INPUT_ZIP does not exist: {p}')
        if not is_phase2_candidate(p): raise RuntimeError(f'Configured INPUT_ZIP is not a complete Phase2 output: {p}')
        return p
    bases=[Path('/kaggle/input'),Path('/kaggle/working'),Path.cwd()]
    candidates=[]
    for base in bases:
        if not base.exists(): continue
        candidates += [p for p in base.rglob('*.zip') if is_phase2_candidate(p)]
        candidates += [p.parent.parent for p in base.rglob('00_config/RUN_MANIFEST.json') if is_phase2_candidate(p.parent.parent)]
    candidates=list(dict.fromkeys(str(p.resolve()) for p in candidates))
    if len(candidates) != 1:
        raise RuntimeError('Need exactly one complete Phase2 results package. Found: '+str(candidates)+'; please set INPUT_ZIP to the exact file or extracted directory.')
    return Path(candidates[0])

archive_path=locate_phase2(); ARC=Phase2Archive(archive_path)
print('Phase 2 input:',archive_path)
print('Files in Phase 2 archive:',len(ARC.list()))
print('Phase 3 output:',OUT)


## 2. Archive integrity and reproducibility

Check ZIP integrity, every indexed file hash, recorded run mode, actual dataset counts and program versions. Missing or mismatched indexed artifacts fail this audit immediately.

In [ ]:
assert all(ARC.exists(x) for x in EXPECTED_FILES), 'Required Phase2 artifact is missing'
zip_status = ARC.zip.testzip() if ARC.is_zip else None
if zip_status is not None: raise RuntimeError(f'Damaged ZIP member: {zip_status}')
RUN = ARC.json('00_config/RUN_MANIFEST.json')
INDEX = ARC.csv('11_reports/ARTIFACT_INDEX.csv')
manifest_checks=[]
for r in INDEX.itertuples(index=False):
    path=str(r.relative_path)
    if not ARC.exists(path):
        manifest_checks.append({'relative_path':path,'status':'MISSING','expected_bytes':r.bytes,'actual_bytes':np.nan,'checksum_match':False})
        continue
    raw=ARC.read(path)
    good_hash=hashlib.sha256(raw).hexdigest()==str(r.sha256)
    good_bytes=len(raw)==int(r.bytes)
    manifest_checks.append({'relative_path':path,'status':'PASS' if good_hash and good_bytes else 'MISMATCH','expected_bytes':int(r.bytes),'actual_bytes':len(raw),'checksum_match':good_hash})
ARTIFACT_CHECKS=pd.DataFrame(manifest_checks)
ARTIFACT_CHECKS.to_csv(OUT/'01_artifact_integrity.csv',index=False)
if (ARTIFACT_CHECKS.status!='PASS').any(): raise RuntimeError('Phase2 file hash/size mismatch! See 01_artifact_integrity.csv')
assert RUN.get('run_mode')=='full', 'This is not a full Phase2 run'
DATA=ARC.csv('01_audit/all_256_original_rows_unchanged.csv')
assert len(DATA)==256 and DATA.orig_id.is_unique, 'Dataset count/IDs inconsistent'
counts=DATA.groupby('temperature_regime').size().to_dict()
assert counts == EXPECTED_ROWS, f'Unexpected regime counts: {counts}'
row_evidence=int(DATA.source_row_evidence.notna().sum())
load_units=int(DATA.filler_loading_unit.notna().sum())
DATA_REG=DATA.set_index('orig_id',drop=False)
INPUT_STATUS=pd.DataFrame([{
    'input_package':str(archive_path), 'n_zip_files':len(ARC.list()),'n_indexed_hashes_verified':len(ARTIFACT_CHECKS),
    'run_mode':RUN.get('run_mode'),'cryogenic_rows':counts['cryogenic'],'room_rows':counts['room'],'high_rows':counts['high'],
    'source_rows_with_independent_evidence':row_evidence,'rows_with_known_loading_units':load_units,
    'scientific_signoff':str(RUN.get('scientific_signoff','UNKNOWN')),
    'verified_publication_level_generalization':False
}])
INPUT_STATUS.to_csv(OUT/'02_input_and_provenance_status.csv',index=False)
print('PASS:',len(ARTIFACT_CHECKS),'artifact hashes; dataset counts',counts)
print('Source rows with evidence:',row_evidence,'/256; loading units specified:',load_units,'/256')


## 3. Rebuild the holdout-prediction ledger without training

Combine original-model predictions and TabICLv2 OOF predictions; independently check unique predictions, identical measured targets, held-out fold labels and original IDs. High-temperature exploratory predictions remain in their **own** table and must not be combined into publication-level CV results.

In [ ]:
ORIG=ARC.csv('05_nested_cv/outer_predictions.csv')
TAB=ARC.csv('05_nested_cv/tabicl_oof_predictions.csv')
HIGH=ARC.csv('05_nested_cv/high_composition_exploratory_predictions.csv')
SPLIT=ARC.csv('03_splits/publication_outer_split_manifest.csv')
OOF=pd.concat([ORIG.assign(model_source='original_phase2'),TAB.assign(model_source='tabicl_addon')],ignore_index=True,sort=False)
assert len(ORIG)>0 and len(TAB)==sum(EXPECTED_ROWS[r] for r in ['cryogenic','room'])
assert set(OOF.regime.unique())=={'cryogenic','room'}
assert OOF[['regime','model','augmentation','orig_id']].duplicated().sum()==0, 'Duplicate OOF prediction for a model/input'
assert (OOF.orig_id.isin(DATA.orig_id)).all()
ref_y=OOF.orig_id.map(DATA_REG.composite_tc_w_mk)
assert np.allclose(OOF.observed.astype(float),ref_y.astype(float),rtol=0,atol=1e-10), 'OOF observed target differs from input data'
assert OOF.predicted.notna().all() and np.isfinite(OOF.predicted.astype(float)).all(), 'Invalid OOF prediction'
# Check that each test prediction actually belongs to a saved outer-fold test partition.
test_manifest=SPLIT[SPLIT.partition.astype(str).str.lower().isin(['test','heldout','validation'])].copy()
assert not test_manifest.empty, 'No held-out partitions in split manifest'
manifest_keys=set(zip(test_manifest.regime,test_manifest.outer_fold.astype(int),test_manifest.orig_id))
pred_keys=set(zip(OOF.regime,OOF.outer_fold.astype(int),OOF.orig_id))
assert pred_keys<=manifest_keys, 'Some predicted observations are not in the reported held-out fold'
# Cross-group split isolation as a COMPUTATIONAL check only.
split_validation=[]
for (reg,fold),part in SPLIT.groupby(['regime','outer_fold']):
    train=part[part.partition.eq('train')]
    test=part[part.partition.eq('test')]
    split_validation.append({'regime':reg,'outer_fold':int(fold),'train_rows':len(train),'test_rows':len(test),
      'same_orig_id_in_train_and_test':len(set(train.orig_id)&set(test.orig_id)),
      'same_declared_source_in_train_and_test':len(set(train.publication_id)&set(test.publication_id)),
      'partition_is_computationally_isolated':not(set(train.orig_id)&set(test.orig_id) or set(train.publication_id)&set(test.publication_id)),
      'publication_labels_independently_verified':False})
SPLIT_AUDIT=pd.DataFrame(split_validation)
SPLIT_AUDIT.to_csv(OUT/'03_split_isolation_audit.csv',index=False)
assert SPLIT_AUDIT.partition_is_computationally_isolated.all(), 'Train/test split isolation failed'
for (reg,model,aug),part in OOF.groupby(['regime','model','augmentation']):
    expected=EXPECTED_ROWS[reg]
    if len(part)!=expected or part.orig_id.nunique()!=expected:raise RuntimeError(f'Incomplete OOF: {reg} {model}/{aug}: {len(part)} rows instead of {expected}')
assert HIGH.regime.eq('high').all()
for model,p in HIGH.groupby('model'):
    assert len(p)==42 and p.orig_id.nunique()==42, f'Incomplete high exploratory predictions: {model}'
    assert np.allclose(p.observed.astype(float),p.orig_id.map(DATA_REG.composite_tc_w_mk).astype(float),rtol=0,atol=1e-10)
OOF.to_csv(OUT/'04_verified_oof_prediction_ledger.csv',index=False)
print('PASS: original OOF rows:',len(ORIG),'TabICLv2 OOF rows:',len(TAB),'high exploratory rows:',len(HIGH))
print('PASS: held-out fold IDs and computational source-label isolation (not verified provenance).')


## 4. Recompute all MAE, RMSE, R², bias and MAPE values

Do not average fold-level R² to estimate pooled R². Independently recompute from the full held-out prediction vector. Compare against the saved Phase 2 metrics; flag differences above numerical tolerance.

In [ ]:
def metrics(y_true,y_pred):
    y=np.asarray(y_true,float);p=np.asarray(y_pred,float)
    if not len(y) or not (np.isfinite(y).all() and np.isfinite(p).all()):raise ValueError('Missing/non-finite metrics inputs')
    err=p-y
    ss=float(np.sum((y-y.mean())**2))
    return {'n':len(y),'mae':float(np.mean(abs(err))), 'rmse':float(np.sqrt(np.mean(err**2))),
      'r2':float(1-np.sum(err**2)/ss) if ss>0 else np.nan,
      'bias':float(np.mean(err)),
      'mape_pct':float(100*np.mean(np.abs(err[np.abs(y)>EPS]/y[np.abs(y)>EPS]))) if (np.abs(y)>EPS).any() else np.nan}
M=[]
for (reg,model,aug),p in OOF.groupby(['regime','model','augmentation']):
    M.append({'regime':reg,'model':model,'augmentation':aug,'validation_type':'PROVISIONAL_SOURCE_GROUPS_NOT_INDEPENDENTLY_VERIFIED',**metrics(p.observed,p.predicted)})
for model,p in HIGH.groupby('model'):
    M.append({'regime':'high','model':model,'augmentation':'none','validation_type':'EXPLORATORY_COMPOSITION_SIGNATURE_NOT_PUBLICATION_INDEPENDENT',**metrics(p.observed,p.predicted)})
RECOMP=pd.DataFrame(M).sort_values(['regime','mae']).reset_index(drop=True)
RECOMP.to_csv(OUT/'05_recalculated_metrics.csv',index=False)
PHASE2_SCORES=ARC.csv('05_nested_cv/oof_overall_metrics.csv')
PHASE2_HIGH=ARC.csv('05_nested_cv/high_composition_exploratory_metrics.csv')
PHASE2_TAB_REPORTED=ARC.csv('11_reports/phase2_all_models_with_exploratory_high.csv')
PHASE2_TAB_REPORTED=PHASE2_TAB_REPORTED[(PHASE2_TAB_REPORTED.model=='TabICLv2') & (PHASE2_TAB_REPORTED.regime.isin(['cryogenic','room']))]
PHASE2_SCORES=pd.concat([PHASE2_SCORES,PHASE2_HIGH,PHASE2_TAB_REPORTED],ignore_index=True,sort=False)
COMPARISON=RECOMP.merge(PHASE2_SCORES[['regime','model','augmentation','n','mae','rmse','r2','bias']],on=['regime','model','augmentation'],how='left',suffixes=('_recomputed','_phase2'),indicator=True)
for k in ['mae','rmse','r2','bias']:
    COMPARISON['abs_delta_'+k]=abs(COMPARISON[k+'_recomputed']-COMPARISON[k+'_phase2'])
COMPARISON['saved_bias_available']=COMPARISON.bias_phase2.notna()
COMPARISON['saved_n_available']=COMPARISON.n_phase2.notna()
COMPARISON['match_all_metrics']=COMPARISON._merge.eq('both')&COMPARISON[[f'abs_delta_{k}' for k in ['mae','rmse','r2']]].le(2e-6).all(axis=1)&(COMPARISON.abs_delta_bias.le(2e-6)|~COMPARISON.saved_bias_available)&(COMPARISON.n_recomputed.eq(COMPARISON.n_phase2)|~COMPARISON.saved_n_available)
COMPARISON.to_csv(OUT/'06_saved_vs_recalculated_metrics.csv',index=False)
if not COMPARISON.match_all_metrics.all():
    raise RuntimeError('One or more reported Phase2 metrics do not match held-out predictions; inspect 06_saved_vs_recalculated_metrics.csv')
print('PASS:',len(COMPARISON),'model/config metric records independently recomputed; bias and n checked where saved (TabICLv2 only has MAE/RMSE/R² saved).')
print(RECOMP[RECOMP.model.eq('TabICLv2')][['regime','n','mae','rmse','r2']].to_string(index=False))


## 5. Reproducible paired group tests with multiplicity control

Exactly paired on the same provisional publication groups, not on individual correlated rows. We report difference in **per-group MAE** (TabICLv2 − previous model), exact two-sided sign-flip p-values, group bootstrap intervals and Holm adjustment. Also audit all pairwise nonaugmented six-model comparisons as a separate descriptive family. **Groups are unverified**, so none is formal publication-level inference.

In [ ]:
def signflip_p(diff):
    a=np.asarray(diff,dtype=float);a=a[np.isfinite(a)]
    if not len(a):return np.nan
    if len(a)>18: raise ValueError('Exact sign flip reserved for <=18 independent units')
    obs=abs(np.mean(a))
    vals=[abs(np.mean(a*np.asarray(s,dtype=float))) for s in itertools.product([-1,1],repeat=len(a))]
    return float(np.mean(np.asarray(vals)>=obs-1e-12))
def group_boot_ci(diff,B=15000):
    a=np.asarray(diff,dtype=float);a=a[np.isfinite(a)]
    if not len(a):return (np.nan,np.nan)
    draws=RNG.choice(a,size=(B,len(a)),replace=True).mean(axis=1)
    return tuple(float(x) for x in np.quantile(draws,[.025,.975]))
def holm_adjust(pvals):
    p=np.asarray(pvals,float)
    out=np.full(len(p),np.nan);valid=np.where(np.isfinite(p))[0]
    order=valid[np.argsort(p[valid])];k=len(order);last=0.0
    for pos,idx in enumerate(order):
        last=max(last,min(1.0,(k-pos)*float(p[idx])))
        out[idx]=last
    return out
# GroupBy.apply above in recent pandas may drop group labels; calculate directly via absolute error instead.
OOF['_abs_err']=abs(OOF.predicted-OOF.observed)
PG=OOF.groupby(['regime','model','augmentation','publication_id'],as_index=False).agg(group_mae=('_abs_err','mean'),n_rows=('orig_id','size'))
PG.to_csv(OUT/'07_per_publication_provisional_mae.csv',index=False)
def paired_groups(reg,model_a,aug_a,model_b,aug_b):
    a=PG[(PG.regime==reg)&(PG.model==model_a)&(PG.augmentation==aug_a)][['publication_id','group_mae']].rename(columns={'group_mae':'mae_a'})
    b=PG[(PG.regime==reg)&(PG.model==model_b)&(PG.augmentation==aug_b)][['publication_id','group_mae']].rename(columns={'group_mae':'mae_b'})
    if a.publication_id.nunique()!=5 or b.publication_id.nunique()!=5:raise RuntimeError(f'Expected exactly five provisional groups for {reg} {model_a}/{model_b}')
    j=a.merge(b,on='publication_id',how='inner',validate='one_to_one')
    if len(j)!=5: raise RuntimeError('Paired source groups mismatch')
    j['delta_a_minus_b']=j.mae_a-j.mae_b
    return j
TARGET_COMPARISONS=[('cryogenic','TabICLv2','none','GradientBoosting','none'),('room','TabICLv2','none','ExtraTrees','gaussian_1pct')]
TARGET_RESULTS=[];GROUP_DIFFS=[]
for reg,a,aa,b,bb in TARGET_COMPARISONS:
    j=paired_groups(reg,a,aa,b,bb);v=j.delta_a_minus_b.to_numpy();lo,hi=group_boot_ci(v)
    TARGET_RESULTS.append({'regime':reg,'candidate':a,'candidate_augmentation':aa,'previous_best':b,'previous_best_augmentation':bb,
       'n_provisional_groups':len(v),'groups_with_lower_candidate_mae':int((v<0).sum()),
       'mean_group_mae_difference_candidate_minus_previous':float(v.mean()),'bootstrap_ci95_low':lo,'bootstrap_ci95_high':hi,
       'p_exact_two_sided_UNADJUSTED':signflip_p(v), 'claim_status':'EXPLORATORY_UNVERIFIED_PUBLICATION_GROUPS'})
    for rr in j.to_dict('records'): GROUP_DIFFS.append({'regime':reg,'candidate':a,'previous_best':b,**rr})
TARGET_RESULTS=pd.DataFrame(TARGET_RESULTS)
TARGET_RESULTS['p_holm_two_specified_posthoc_comparisons']=holm_adjust(TARGET_RESULTS.p_exact_two_sided_UNADJUSTED)
TARGET_RESULTS.to_csv(OUT/'08_targeted_tabicl_vs_previous_significance.csv',index=False)
pd.DataFrame(GROUP_DIFFS).to_csv(OUT/'09_targeted_group_differences.csv',index=False)
# Six primary models only; no augmentation to avoid mixing model and augmentation effects. Pair selection is post-hoc and descriptive.
PAIRWISE=[]
for reg in ['cryogenic','room']:
    candidates=sorted(set(PG[(PG.regime==reg)&(PG.augmentation=='none')].model.unique()) & set(['ExtraTrees','GradientBoosting','RandomForest','SVR','XGBoost','TabICLv2']))
    for a,b in combinations(candidates,2):
        j=paired_groups(reg,a,'none',b,'none');v=j.delta_a_minus_b.to_numpy();lo,hi=group_boot_ci(v,B=5000)
        PAIRWISE.append({'regime':reg,'model_a':a,'model_b':b,'n_provisional_groups':len(v),
          'mean_delta_group_MAE_a_minus_b':float(v.mean()),'bootstrap_ci95_low':lo,'bootstrap_ci95_high':hi,
          'p_unadjusted':signflip_p(v)})
PAIRWISE=pd.DataFrame(PAIRWISE)
for reg,ind in PAIRWISE.groupby('regime').groups.items():
    PAIRWISE.loc[ind,'p_holm_within_regime_family']=holm_adjust(PAIRWISE.loc[ind,'p_unadjusted'])
PAIRWISE['scientific_status']='PROVISIONAL_GROUPS; NO VERIFIED_SOURCE_GENERALIZATION'
PAIRWISE.to_csv(OUT/'10_six_model_pairwise_significance.csv',index=False)
print(TARGET_RESULTS[['regime','groups_with_lower_candidate_mae','mean_group_mae_difference_candidate_minus_previous','p_exact_two_sided_UNADJUSTED','p_holm_two_specified_posthoc_comparisons']].to_string(index=False))
print('Minimum attainable two-sided exact sign-flip p with five nonzero paired groups =',2/2**5)


## 6. Augmentation ablation and training-only safety checks

Compare zero synthetic training copies against 1% and historical-strength 3% experiments using the same held-out groups. Generated rows are not independent experiments. This notebook does **not** claim that the original historical augmentation procedure is exactly reproduced.

In [ ]:
AUG_ROWS=[]
for reg in ['cryogenic','room']:
    models=sorted(set(OOF[(OOF.regime==reg)&(OOF.augmentation=='none')].model)-{'TabICLv2','train_mean','matrix_k'})
    for model in models:
        for aug in ['gaussian_1pct','gaussian_3pct_historical']:
            if not ((OOF.regime==reg)&(OOF.model==model)&(OOF.augmentation==aug)).any(): continue
            j=paired_groups(reg,model,aug,model,'none');d=j.delta_a_minus_b.to_numpy();lo,hi=group_boot_ci(d,B=8000)
            AUG_ROWS.append({'regime':reg,'model':model,'augmentation':aug,'n_groups':len(d),
                'mean_group_delta_mae_aug_minus_none':float(d.mean()),'groups_improved':int((d<0).sum()),
                'ci95_low':lo,'ci95_high':hi,'p_two_sided_UNADJUSTED':signflip_p(d),
                'training_only_two_synthetic_copies':True,
                'not_new_independent_measurements':True})
AUG=pd.DataFrame(AUG_ROWS)
for reg,ind in AUG.groupby('regime').groups.items():
    AUG.loc[ind,'p_holm_within_regime_10_tests']=holm_adjust(AUG.loc[ind,'p_two_sided_UNADJUSTED'])
AUG['claim_status']='INPUT_PERTURBATION_SENSITIVITY_ONLY; GROUPS_UNVERIFIED'
AUG.to_csv(OUT/'11_augmentation_recomputed_ablation.csv',index=False)
# Inspect recorded experiment design, not unverifiable source code execution from metrics alone.
CONFIG=ARC.json('00_config/runtime_config.json')
AUG_PROTOCOL=pd.DataFrame([{'reported_augmentation_sd_1pct':RUN.get('augmentation_input_only_sd_relative'),
 'historical_3pct_sd':RUN.get('historical_implemented_noise_sd_relative'),
 'copies_per_training_original':RUN.get('augmentation_train_copies'),
 'train_heldout_ID_overlap_count':int(SPLIT_AUDIT.same_orig_id_in_train_and_test.sum()),
 'train_heldout_label_overlap_count':int(SPLIT_AUDIT.same_declared_source_in_train_and_test.sum()),
 'can_independently_verify_actual_train_only_noise_from_saved_predictions':False,
 'conclusion':'Split manifests support isolation; pipeline source code plus provenance required to certify augmentation implementation.'}])
AUG_PROTOCOL.to_csv(OUT/'12_augmentation_protocol_limits.csv',index=False)
print('Ablation comparisons:',len(AUG),'| No claim that synthetic rows are new observations.')


## 7. Worst predictions, model bias and error concentration

Keep actual recorded predictions, label numerical outliers only as diagnostic flags, and report errors by observed-conductivity band, filler-loading band, and measured-temperature band where available. **Do not remove poorly predicted observations.**

In [ ]:
DIAG=OOF.copy()
DIAG['signed_error_W_mK']=DIAG.predicted-DIAG.observed
DIAG['abs_error_W_mK']=abs(DIAG.signed_error_W_mK)
DIAG['relative_error_pct']=100*DIAG.abs_error_W_mK/DIAG.observed.abs().where(DIAG.observed.abs()>EPS,np.nan)
for k in ['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded','composite_tc_w_mk']:
    DIAG[k]=DIAG.orig_id.map(DATA_REG[k])
ALL_DIAG=pd.concat([DIAG,HIGH.assign(augmentation='none',publication_id='NOT_INDEPENDENT',signed_error_W_mK=HIGH.predicted-HIGH.observed,
                         abs_error_W_mK=abs(HIGH.predicted-HIGH.observed),
                         relative_error_pct=100*abs(HIGH.predicted-HIGH.observed)/HIGH.observed.abs().where(HIGH.observed.abs()>EPS,np.nan))],ignore_index=True,sort=False)
for k in ['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded','composite_tc_w_mk']:
    ALL_DIAG[k]=ALL_DIAG.orig_id.map(DATA_REG[k])
ALL_DIAG['evaluation_type']=np.where(ALL_DIAG.regime.eq('high'),'EXPLORATORY_COMPOSITION_SIGNATURE','PROVISIONAL_SOURCE_GROUPS')
ALL_DIAG.sort_values('abs_error_W_mK',ascending=False).head(100).to_csv(OUT/'13_worst_100_predictions.csv',index=False)
ALL_DIAG[(ALL_DIAG.model.eq('TabICLv2'))].sort_values('abs_error_W_mK',ascending=False).head(30).to_csv(OUT/'14_worst_tabicl_predictions.csv',index=False)
INVALID=ALL_DIAG[(ALL_DIAG.predicted<=0)|(ALL_DIAG.observed<=0)].copy()
INVALID.to_csv(OUT/'15_nonpositive_thermal_conductivity_flags.csv',index=False)
# Quantile bins describe test-set error concentration, not a training-time data transformation.
SEG_ROWS=[]
for (reg,model,aug),p in ALL_DIAG.groupby(['regime','model','augmentation']):
    if model not in ('TabICLv2','GradientBoosting','ExtraTrees','SVR'): continue
    for field in ['composite_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded']:
        vals=pd.to_numeric(p[field],errors='coerce')
        if vals.notna().sum()<6 or vals.nunique()<3: continue
        q=pd.qcut(vals,q=min(4,vals.nunique()),duplicates='drop')
        for label,ss in p.groupby(q,observed=True):
            if not len(ss):continue
            SEG_ROWS.append({'regime':reg,'model':model,'augmentation':aug,'feature':field,'quantile_band':str(label),
              'n':len(ss),'mae_W_mK':float(ss.abs_error_W_mK.mean()),'bias_W_mK':float(ss.signed_error_W_mK.mean())})
SEG=pd.DataFrame(SEG_ROWS)
SEG.to_csv(OUT/'16_error_by_recorded_variable_band.csv',index=False)
ERROR_SUMMARY=ALL_DIAG.groupby(['regime','model','augmentation'],as_index=False).agg(
 n=('orig_id','size'), mean_signed_error=('signed_error_W_mK','mean'),median_abs_error=('abs_error_W_mK','median'),
 p90_abs_error=('abs_error_W_mK',lambda s:float(np.quantile(s,.9))),max_abs_error=('abs_error_W_mK','max'),
 count_nonpositive_pred=('predicted',lambda s:int((s<=0).sum())))
ERROR_SUMMARY.to_csv(OUT/'17_bias_outlier_summaries.csv',index=False)
print('Nonpositive model predictions:',len(INVALID),'(may repeat same original observation across configurations)')
print('Worst TabICLv2 records:')
print(ALL_DIAG[ALL_DIAG.model.eq('TabICLv2')].sort_values('abs_error_W_mK',ascending=False)[['regime','orig_id','observed','predicted','abs_error_W_mK']].head(8).to_string(index=False))


## 8. Prediction-interval audit (not laboratory uncertainty)

Recompute empirical interval coverage and widths, check bounds and claimed guarantee, and report whether nominal 90% interval performance was achieved on the saved records. Because distributions are heterogeneous and source labels unverified, these are empirical diagnostics only; do not claim 90% distribution-free coverage for unseen publications.

In [ ]:
CONF=ARC.csv('10_uncertainty/heldout_group_conformal_predictions.csv')
CONF_GROUPS=ARC.csv('10_uncertainty/heldout_group_interval_coverage.csv')
assert len(CONF)==214, 'Unexpected Phase2 conformal record count'
CONF['covered_recomputed']=(CONF.observed>=CONF.lower_90pct)&(CONF.observed<=CONF.upper_90pct)
CONF['width_recomputed']=CONF.upper_90pct-CONF.lower_90pct
assert (CONF.width_recomputed>=0).all(), 'Invalid negative interval width'
assert (CONF.covered_recomputed==CONF.covered.astype(bool)).all(), 'Saved covered flags inconsistent'
assert np.allclose(CONF.width_recomputed,CONF.width,rtol=0,atol=2e-6)
interval_rows=[]
for reg,p in CONF.groupby('regime'):
    interval_rows.append({'regime':reg,'n_predictions':len(p),'empirical_coverage':p.covered_recomputed.mean(),
      'nominal_coverage_reference':.90,'mean_interval_width_W_mK':p.width_recomputed.mean(),
      'negative_lower_bound_count':int((p.lower_90pct<0).sum()),
      'calibration_publication_groups':p.cal_publication.nunique(),
      'coverage_guarantee_for_unseen_publications':'NONE','measurement_uncertainty_estimated':False})
COVERAGE=pd.DataFrame(interval_rows)
COVERAGE.to_csv(OUT/'18_empirical_interval_coverage.csv',index=False)
CONF.to_csv(OUT/'19_verified_interval_ledger.csv',index=False)
CONF_GROUPS.to_csv(OUT/'20_original_per_group_interval_coverage.csv',index=False)
print(COVERAGE[['regime','n_predictions','empirical_coverage','mean_interval_width_W_mK','negative_lower_bound_count']].to_string(index=False))


## 9. Source data, physical baselines, measurement and external-validation gates

Do not silently convert `filler_loading_as_recorded` to volume fraction or infer material identities. Where physical-model comparisons, Fourier-law uncertainty, calibration or verified external testing have no eligible rows, report **BLOCKED**, not zero error or an empty successful test.

In [ ]:
EMPTY_REQUIREMENTS={
 'strict_epoxy_nested_metrics':'06_stats/strict_epoxy_nested_metrics.csv',
 'material_grouped_nested_cv':'06_stats/material_grouped_nested_cv.csv',
 'verified_external_metrics':'08_external/verified_external_metrics.csv',
 'same_row_ml_vs_effective_medium':'09_physics/same_row_ml_vs_effective_medium.csv',
 'fourier_law_measurements':'10_uncertainty/fourier_law_measurements.csv',
 'standard_reference_calibration_checks':'10_uncertainty/standard_reference_calibration_checks.csv',
}
GATES=[]
for name,rel in EMPTY_REQUIREMENTS.items():
    d=ARC.csv(rel,allow_empty=True) if ARC.exists(rel) else pd.DataFrame()
    GATES.append({'evidence_requirement':name,'phase2_rows':len(d),
       'status':'BLOCKED_MISSING_VERIFIED_EVIDENCE' if d.empty else 'REQUIRES_INDEPENDENT_SCIENTIFIC_REVIEW',
       'not_equivalent_to_an_error_free_result':True})
GATES += [
 {'evidence_requirement':'source_to_row_matching','phase2_rows':row_evidence,'status':'BLOCKED_0_OF_256_VERIFIED','not_equivalent_to_an_error_free_result':True},
 {'evidence_requirement':'filler_loading_unit','phase2_rows':load_units,'status':'BLOCKED_0_OF_256_DOCUMENTED','not_equivalent_to_an_error_free_result':True},
 {'evidence_requirement':'declared_high_independent_publications','phase2_rows':int(DATA[DATA.temperature_regime=='high'].publication_id.nunique()),'status':'BLOCKED_ONE_PROVISIONAL_SOURCE','not_equivalent_to_an_error_free_result':True},
 {'evidence_requirement':'manuscript_259_vs_available_256','phase2_rows':3,'status':'UNRESOLVED_DIFFERENCE_OF_3_ROWS','not_equivalent_to_an_error_free_result':True},
]
GATES=pd.DataFrame(GATES)
GATES.to_csv(OUT/'21_scientific_evidence_gates.csv',index=False)
# Keep independent source overlap/physical quality flags as evidence not fabricated conversions.
QUALITY=ARC.csv('01_audit/row_quality_flags.csv')
QUALITY.to_csv(OUT/'22_phase2_row_quality_flags_copied.csv',index=False)
FEASIBILITY=ARC.csv('03_splits/group_cv_feasibility.csv')
FEASIBILITY.to_csv(OUT/'23_group_cv_feasibility_copied.csv',index=False)
# Existing feature-importance evidence is descriptive, never causal verification.
PERM_EVIDENCE=ARC.csv('07_interpretation/heldout_group_permutation_importance.csv')
SHAP_EVIDENCE=ARC.csv('07_interpretation/mean_absolute_shap.csv')
FEATURE_AUDIT=pd.DataFrame([
 {'evidence_type':'heldout_group_permutation_importance','n_records':len(PERM_EVIDENCE),'scientific_interpretation':'DESCRIPTIVE_ASSOCIATION_ONLY','is_causal_evidence':False},
 {'evidence_type':'mean_absolute_shap','n_records':len(SHAP_EVIDENCE),'scientific_interpretation':'DESCRIPTIVE_ASSOCIATION_ONLY','is_causal_evidence':False}
])
FEATURE_AUDIT.to_csv(OUT/'30_feature_importance_claim_audit.csv',index=False)
# Check tuning records against outer fold scores. Inner/outer gaps are diagnostic, not independent train-vs-test bias estimates.
INNER=ARC.csv('05_nested_cv/inner_tuning_best.csv')
FOLDS=ARC.csv('05_nested_cv/outer_fold_metrics.csv')
TUNING_GAP=INNER.merge(FOLDS[['regime','outer_fold','model','augmentation','mae']],on=['regime','outer_fold','model','augmentation'],how='left',validate='one_to_one')
TUNING_GAP['outer_mae_minus_inner_cv_mae']=TUNING_GAP.mae-TUNING_GAP.best_inner_mae
TUNING_GAP['interpretation']='TRAIN_SIDE_INNER_CV_VERSUS_HELDOUT_GROUP_DIAGNOSTIC_NOT_TRAIN_ERROR'
TUNING_GAP.to_csv(OUT/'31_inner_cv_vs_outer_fold_diagnostics.csv',index=False)
print(GATES[['evidence_requirement','status']].to_string(index=False))


## 10. Reviewer-by-reviewer evidence matrix (all 40 comments)

The reviewer register is copied from Phase 2; it is **not** a declaration of acceptance by the journal. Map every comment to available verification or necessary outside evidence. Comments about manuscript editing/SEM/cure schedules remain manuscript/laboratory tasks.

In [ ]:
ISSUES=ARC.csv('11_reports/REVIEWER_40_ISSUE_STATUS.csv')
assert len(ISSUES)==40 and set(ISSUES.reviewer)=={'R1','R2','R3'}
REVIEW_MAP={
'R1':{1:('A1_A4','CONDITIONAL'),2:('A1_A5','BLOCKED'),3:('A1_A2','CONDITIONAL'),4:('A2_A3','CONDITIONAL'),5:('A6_A8','BLOCKED'),6:('A1_A5','BLOCKED'),7:('A5_A8','BLOCKED'),8:('A1','CONDITIONAL'),9:('A5_A7','BLOCKED'),10:('A8','BLOCKED'),11:('A5_A8','CONDITIONAL'),12:('A7','BLOCKED'),13:('A8','BLOCKED'),14:('MANUSCRIPT_LAB','OUT_OF_SCOPE'),15:('A8','CONDITIONAL'),16:('A1_A2','CONDITIONAL'),17:('MANUSCRIPT','OUT_OF_SCOPE')},
'R2':{1:('A1_A4','CONDITIONAL'),2:('A1_A5','BLOCKED'),3:('A1_A5','BLOCKED'),4:('A5','CONDITIONAL'),5:('MANUSCRIPT','OUT_OF_SCOPE'),6:('LAB','BLOCKED'),7:('A7','BLOCKED'),8:('LAB','BLOCKED'),9:('A7','BLOCKED'),10:('A7','BLOCKED'),11:('A8','BLOCKED'),12:('MANUSCRIPT','OUT_OF_SCOPE'),13:('A4','CONDITIONAL'),14:('A8','CONDITIONAL'),15:('A2_A3','CONDITIONAL'),16:('A8','BLOCKED'),17:('A5_A7','BLOCKED')},
'R3':{1:('MANUSCRIPT','OUT_OF_SCOPE'),2:('A1_A4','CONDITIONAL'),3:('A1_A5','BLOCKED'),4:('A7','BLOCKED'),5:('A2_A3_A7','CONDITIONAL'),6:('MANUSCRIPT','OUT_OF_SCOPE')}
}
REVIEW_ROWS=[]
for r in ISSUES.itertuples(index=False):
    key=(str(r.reviewer),int(r.comment_number));audit,assess=REVIEW_MAP[key[0]][key[1]]
    REVIEW_ROWS.append({**r._asdict(),'phase3_audit_task':audit,'phase3_evidence_status':assess,
       'all_computed_results_are_provisional_when_group_based':True,
       'reviewer_satisfied_by_phase3_alone':False,
       'next_action':('Manuscript/lab evidence needed' if assess in ('BLOCKED','OUT_OF_SCOPE') else 'Use recomputed tables with explicit methodological caveats')})
REVIEW=pd.DataFrame(REVIEW_ROWS)
REVIEW.to_csv(OUT/'24_reviewer_40_evidence_matrix.csv',index=False)
print('All reviewer comments mapped:',len(REVIEW),REVIEW.phase3_evidence_status.value_counts().to_dict())


## 11. Publication-oriented figures and audit report

Figures are labelled **provisional** or **exploratory**. The report separates actual error reductions from statistical significance, and observational intervals from experimental measurement uncertainty.

In [ ]:
# Plot 1: all six models, using no-augmentation wherever possible; compare original best augmentation separately.
sel=RECOMP[(RECOMP.regime.isin(['cryogenic','room']))&(RECOMP.model.isin(['ExtraTrees','GradientBoosting','RandomForest','SVR','XGBoost','TabICLv2']))&(RECOMP.augmentation=='none')]
fig,axs=plt.subplots(1,2,figsize=(12,4.3))
for ax,reg in zip(axs,['cryogenic','room']):
    d=sel[sel.regime==reg].sort_values('mae')
    ax.barh(d.model,d.mae)
    ax.invert_yaxis();ax.set_title(f'{reg.title()}: provisional source groups');ax.set_xlabel('OOF MAE (W/(m·K))')
    for j,(_,r) in enumerate(d.iterrows()):ax.text(r.mae,j,f' {r.mae:.3f}',va='center',fontsize=8)
fig.suptitle('All six models — no augmentation (not verified independent publications)')
fig.tight_layout();fig.savefig(OUT/'25_six_models_mae.png',dpi=170,bbox_inches='tight');plt.close(fig)
# Plot 2: best baseline vs TabICLv2 per-regime paired publication-level MAE.
GD=pd.DataFrame(GROUP_DIFFS)
fig,axs=plt.subplots(1,2,figsize=(11,3.8))
for ax,reg in zip(axs,['cryogenic','room']):
    d=GD[GD.regime==reg].sort_values('publication_id');ax.axhline(0,ls='--',color='gray',lw=1)
    ax.bar(d.publication_id,d.delta_a_minus_b);ax.set_title(f'{reg.title()}: TabICLv2 minus previous best')
    ax.set_ylabel('Paired publication-group MAE difference');ax.tick_params(axis='x',rotation=25)
fig.suptitle('Negative = TabICLv2 improved (groups are provisional)')
fig.tight_layout();fig.savefig(OUT/'26_paired_group_mae_deltas.png',dpi=170,bbox_inches='tight');plt.close(fig)
# Plot 3: TabICLv2 held-out predictions and errors.
fig,axs=plt.subplots(1,3,figsize=(13,4))
for ax,reg in zip(axs,['cryogenic','room','high']):
    d=ALL_DIAG[(ALL_DIAG.regime==reg)&(ALL_DIAG.model=='TabICLv2')]
    ax.scatter(d.observed,d.predicted,s=17,alpha=.7)
    xy0=min(d.observed.min(),d.predicted.min());xy1=max(d.observed.max(),d.predicted.max());ax.plot([xy0,xy1],[xy0,xy1],linestyle='--',color='gray')
    ax.set_title(f'{reg.title()}'+(' (exploratory)' if reg=='high' else ' (provisional)'))
    ax.set_xlabel('Observed TC (W/(m·K))');ax.set_ylabel('Predicted TC (W/(m·K))')
fig.tight_layout();fig.savefig(OUT/'27_tabicl_observed_vs_predicted.png',dpi=170,bbox_inches='tight');plt.close(fig)
# Plot 4: interval coverage.
fig,ax=plt.subplots(figsize=(6,4))
ax.bar(COVERAGE.regime,COVERAGE.empirical_coverage);ax.axhline(.9,ls='--',color='gray',label='90% nominal reference')
ax.set_ylim(0,1.02);ax.set_ylabel('Empirical held-out interval coverage');ax.set_title('No guarantee for independent publications');ax.legend()
fig.tight_layout();fig.savefig(OUT/'28_empirical_interval_coverage.png',dpi=170,bbox_inches='tight');plt.close(fig)

TAB_REPORT=RECOMP[RECOMP.model.eq('TabICLv2')].set_index('regime')
PRIOR=ARC.csv('11_reports/phase2_original_vs_tabicl_comparison.csv')
report=[]
report.append('# Phase 3 statistical audit — Verified numerical outputs, conditional scientific conclusions')
report.append(f'\nSource: `{archive_path}`; audit UTC: {datetime.now(timezone.utc).isoformat()}')
report.append(f'\n- Archive: {len(ARC.list())} files, {len(ARTIFACT_CHECKS)} indexed hashes verified, no ZIP corruption.')
report.append(f'- Original records: {counts}; source-row evidence: {row_evidence}/256; documented loading units: {load_units}/256.')
report.append(f'- Original CV predictions: {len(ORIG)}; TabICLv2 CV predictions: {len(TAB)}; high composition exploratory: {len(HIGH)}.')
report.append(f'- {len(COMPARISON)} pooled model metric records independently reproduced; MAE/RMSE/R² all match within 2e-6, and n/bias checked for the {int(COMPARISON.saved_bias_available.sum())} rows where provided.')
report.append('\n## Results comparison (observational, provisional)')
for r in PRIOR.itertuples(index=False):
    v=TAB_REPORT.loc[r.regime]
    report.append(f'- **{r.regime}:** original winner {r.previous_best_model}/{r.previous_best_augmentation} MAE {r.previous_best_mae:.5f}; TabICLv2 MAE {v.mae:.5f}; MAE reduction {r.percent_mae_reduction:.1f}%; TabICLv2 R² {v.r2:.4f}.')
for r in TARGET_RESULTS.itertuples(index=False):
    report.append(f'- {r.regime} TabICLv2 vs previous winner: {r.groups_with_lower_candidate_mae}/5 provisional groups improved, two-sided exact sign-flip p={r.p_exact_two_sided_UNADJUSTED:.4f}, Holm (two specified post-hoc comparisons) p={r.p_holm_two_specified_posthoc_comparisons:.4f}.')
report.append('The small number of independent *verified* groups is unknown. Under five non-zero paired units the minimum attainable exact two-sided p-value is 0.0625. No significance or independent-source superiority is established.')
if 'high' in TAB_REPORT.index:
    h=TAB_REPORT.loc['high']
    report.append(f'\nHigh-temperature TabICLv2: exploratory MAE {h.mae:.5f}, R² {h.r2:.4f} on {int(h.n)} held-out observations. There is only one declared publication and 40 numerical composition signatures; these results do NOT establish independent-material/publication generalization.')
report.append('\n## Verification status')
report.append(f'- Paired tests recalculated: {len(PAIRWISE)} six-model unaugmented group comparisons and {len(AUG)} augmentation ablations, plus 2 specified post-hoc baseline comparisons.')
report.append(f'- Nonpositive predicted conductivity instances across all models/configurations: {len(INVALID)}. These require physical-validity commentary.')
report.append('- Predictions intervals: empirical coverage only, not ISO GUM laboratory uncertainty nor guaranteed unseen-source coverage.')
report.append('- The paper reports 259 observations; the supplied original-data audit contains 256 (difference of 3 unresolved).')
report.append('- Source assignments were author confirmed but originated from earlier random placeholders; independent source mapping is required before publication-level claims.')
report.append('- Physical effective-medium comparison is blocked because filler-loading units, material identity and individual source evidence remain unverified.')
report.append('- Calibration records, specimen-level replicates, Fourier-law measurement inputs and independent external experiments are not included in the input ZIP.')
report.append('\n## Reviewer-aligned action items')
report.append('1. Replace original leakage-prone random-split results with correctly labeled computational evidence; source-verification is still required.')
report.append('2. Describe TabICLv2 as lower observed MAE, **not statistically proven superior**.')
report.append('3. Separate exploratory high-temperature scores from independently held-out evaluation.')
report.append('4. Resolve data count, publication-to-row mapping, filler units and true epoxy-only cohort; re-evaluate affected results only if source corrections change data or groups.')
report.append('5. Obtain original lab measurement tables and instrument calibration/uncertainty budget to answer Measurement-specific reviewer concerns.')
report.append('6. Write response-letter tables with specific caveats; do not overstate external experimental validation, physical mechanism or space-grade readiness.')
(OUT/'PHASE3_STATISTICAL_AUDIT_REPORT.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
print('Created Phase 3 statistical audit report and four figures.')


## 12. Output manifest, completeness checks, and ZIP

All Phase 3 output files, including reviewer evidence and audit limitations, are checksummed and zipped. This archive is the only file you need to return for the follow-up scientific review.

In [ ]:
# Require core outputs to exist before declaring a completed audit.
REQUIRED_PHASE3=['01_artifact_integrity.csv','05_recalculated_metrics.csv','06_saved_vs_recalculated_metrics.csv',
 '08_targeted_tabicl_vs_previous_significance.csv','10_six_model_pairwise_significance.csv',
 '11_augmentation_recomputed_ablation.csv','14_worst_tabicl_predictions.csv',
 '18_empirical_interval_coverage.csv','21_scientific_evidence_gates.csv',
 '24_reviewer_40_evidence_matrix.csv','PHASE3_STATISTICAL_AUDIT_REPORT.md']
missing=[x for x in REQUIRED_PHASE3 if not (OUT/x).exists()]
if missing:raise RuntimeError('Missing Phase 3 outputs: '+str(missing))
STATUS={
 'run_status':'COMPLETE_COMPUTATIONAL_AUDIT_WITH_SCIENTIFIC_BLOCKERS',
 'archive_integrity':'PASS', 'phase2_hashes_verified':len(ARTIFACT_CHECKS),
 'recomputed_metric_records':len(COMPARISON), 'metric_reproduction_all_match':bool(COMPARISON.match_all_metrics.all()),
 'n_original_observations':len(DATA), 'n_oof_original':len(ORIG),'n_oof_tabicl':len(TAB),'n_high_exploratory_predictions':len(HIGH),
 'paired_model_tests':len(PAIRWISE),'augmentation_tests':len(AUG),'reviewer_comments_mapped':len(REVIEW),
 'scientific_signoff':'BLOCKED: SOURCE_PROVENANCE_UNVERIFIED; LOADING_UNITS_AND_LAB_DATA_MISSING',
 'no_model_training':True, 'input_run_mode':RUN.get('run_mode'),
 'output_generated_utc':datetime.now(timezone.utc).isoformat()
}
(OUT/'00_AUDIT_STATUS.json').write_text(json.dumps(STATUS,indent=2),encoding='utf-8')
manifest=[]
for f in sorted(OUT.iterdir()):
    if f.is_file():
        b=f.read_bytes();manifest.append({'filename':f.name,'bytes':len(b),'sha256':hashlib.sha256(b).hexdigest()})
pd.DataFrame(manifest).to_csv(OUT/'29_output_file_manifest.csv',index=False)
ZIPPATH=OUTPUT_BASE/'TC_PHASE3_STATISTICAL_AUDIT.zip'
with zipfile.ZipFile(ZIPPATH,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname='TC_PHASE3_STATISTICAL_AUDIT/'+f.name)
with zipfile.ZipFile(ZIPPATH) as z:
    if z.testzip() is not None: raise RuntimeError('Phase3 output ZIP integrity failed')
print('PHASE 3 COMPUTATIONAL AUDIT FINISHED')
print(json.dumps(STATUS,indent=2))
print('ZIP:',ZIPPATH,'size (MB):',round(ZIPPATH.stat().st_size/1e6,3))
